# Урок 05 - Агентски RAG


## Настройка

Този тетрадка демонстрира агентния RAG (Retrieval-Augmented Generation) модел, използвайки Microsoft Agent Framework.

**Предварителни изисквания:**
- `AZURE_AI_PROJECT_ENDPOINT` — вашият крайна точка на Microsoft Foundry проекта
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — името на вашето разгръщане на модела (напр. `gpt-5-mini`)
- Azure CLI с удостоверяване (`az login`)

> **Забележка:** Тази тетрадка използва in-memory база знания, за да можете да се концентрирате върху самия агентен RAG модел — не е необходим ресурс Azure AI Search. За да използвате същия модел с реален индекс в Azure AI Search (както бихте направили в продукция), вижте опционалното [Ръководство за настройка на Azure AI Search](../../00-course-setup/AzureSearch.md).


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Какво е Agentic RAG?

Традиционният RAG следва фиксиран процес: извличане на документи, след което генериране на отговор. **Agentic RAG** отива по-далеч, като дава на агента автономия да решава **кога** и **как** да извлича информация.

С Agentic RAG агентът може:
- **Да реши** дали е необходимо извличане преди да отговори на въпрос
- **Да избере** кой източник на данни или инструмент да използва за заявка
- **Да оцени** извлечените резултати и да направи допълнителни извличания, ако първият опит не е достатъчен
- **Да комбинира** информация от няколко извличания в един последователен отговор

Това прави агента по-гъвкав и точен в сравнение със статичния процес „извличане, после генериране“.


## Създаване на инструмент за търсене

В Agentic RAG външните източници на данни са обвити като **инструменти**, които агентът може да извиква при нужда. Това позволява на агента да третира извличането като просто още едно действие, което може да извърши, а не като задължителна стъпка.

По-долу дефинираме база знания за пътувания и я предоставяме като инструмент, който агентът може да използва за търсене на информация за дестинации.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Създаване на RAG агент

Сега създаваме агент, който е инструктиран **винаги да извлича информация преди да отговори**. Агентът използва инструмента `search_travel_knowledge`, за да базира отговорите си на базата знания, вместо да разчита на собствените си обучителни данни.


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Итеративно извличане — Патърнът Производител-Проверител

Ключово преимущество на Agentic RAG е **итеративното извличане**. Агентът може да извърши няколко кръга търсене, за да потвърди, уточни или разшири първоначалните си находки — подобно на работен процес "производител-проверител":

1. **Стъпка производител**: Агентът извлича първоначална информация и изготвя отговор.
2. **Стъпка проверител**: Агентът извършва допълнителни извличания, за да провери детайли или да запълни пропуски.

По-долу на агента е зададен въпрос, който изисква сравняване на няколко дестинации, подтиквайки го да търси няколко пъти.


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Обобщение

В този урок научихте как да изградите **Agentic RAG** система, използвайки Microsoft Agent Framework:

- **Agentic RAG** позволява на агентите автономно да решават кога да извличат информация, правейки извличането динамично, а не фиксирано.
- **Инструменти като източници на данни**: Външни бази от знания (като Azure AI Search) са обвити като инструменти, които агентът може да използва.
- **Итеративно извличане**: Моделът maker-checker позволява на агента да извършва няколко кръга на извличане — търсене, проверка и усъвършенстване — преди да предостави окончателен отговор.

В продукционна среда трябва да замените in-memory `TRAVEL_KNOWLEDGE_BASE` с реален индекс Azure AI Search за обработка на голям обем от пътеводни документи.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:
Този документ е преведен с помощта на AI преводачески услуга [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, моля имайте предвид, че автоматизираните преводи могат да съдържат грешки или неточности. Оригиналният документ на неговия роден език трябва да се счита за авторитетен източник. За критична информация се препоръчва професионален човешки превод. Ние не носим отговорност за каквито и да е недоразумения или неправилни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
